# Average Treatment Effect Estimators

This notebook estimates the average causal effect of the retention offer using:

- a naive difference in observed outcomes,
- inverse probability weighting,
- a T-learner,
- and cross-fitted augmented inverse probability weighting.

The estimators use only observed variables. The simulated causal ground truth is loaded separately and used only for evaluation.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


current_path = Path.cwd().resolve()

if (current_path / "README.md").exists():
    PROJECT_ROOT = current_path
else:
    PROJECT_ROOT = current_path.parent


OBSERVED_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "customer_retention_observed.csv"
)

FULL_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "customer_retention_full.csv"
)


observed_df = pd.read_csv(OBSERVED_DATA_PATH)
full_df = pd.read_csv(FULL_DATA_PATH)


print("Observed data shape:", observed_df.shape)
print("Full data shape:", full_df.shape)

print(
    "Observed missing values:",
    observed_df.isna().sum().sum(),
)

print(
    "Full missing values:",
    full_df.isna().sum().sum(),
)

print(
    "Row alignment preserved:",
    observed_df.index.equals(full_df.index),
)

display(observed_df.head())

Observed data shape: (6000, 9)
Full data shape: (6000, 19)
Observed missing values: 0
Full missing values: 0
Row alignment preserved: True


,age,tenure,spend,complaints,satisfaction,engagement,region,treatment,outcome
0,47.656605,39.445856,80.538653,1,0.739655,0.149673,South,0,0
1,31.520191,59.643340,124.772237,0,0.621712,0.503829,West,0,0
2,53.005414,14.470515,55.084554,0,0.748244,0.878509,South,1,0
3,55.286777,29.174800,50.544936,1,0.754126,0.605190,East,1,0
4,20.587578,7.246782,47.698135,2,0.532050,0.356345,North,0,0


## 2. Simulated Causal Ground Truth

The full simulated dataset contains both potential outcomes and the true conditional treatment effect.

These hidden quantities are not available to the causal estimators. They are used only to evaluate estimation error.

In [2]:
ground_truth_columns = [
    "true_propensity",
    "p0",
    "p1",
    "true_cate",
    "y0",
    "y1",
]

leaked_columns = sorted(
    set(ground_truth_columns).intersection(
        observed_df.columns
    )
)

assert leaked_columns == []


true_probability_ate = full_df[
    "true_cate"
].mean()

true_probability_ate_check = (
    full_df["p1"] - full_df["p0"]
).mean()

true_binary_ate = (
    full_df["y1"] - full_df["y0"]
).mean()


assert np.isclose(
    true_probability_ate,
    true_probability_ate_check,
)


print(
    "Ground-truth columns in observed data:",
    leaked_columns,
)

print(
    "True probability-scale ATE:",
    true_probability_ate,
)

print(
    "ATE from p1 minus p0:",
    true_probability_ate_check,
)

print(
    "True binary potential-outcome ATE:",
    true_binary_ate,
)

Ground-truth columns in observed data: []
True probability-scale ATE: 0.15836461343754135
ATE from p1 minus p0: 0.1583646134375414
True binary potential-outcome ATE: 0.14883333333333335


## 3. Naive Difference in Observed Outcomes

The naive estimator subtracts the observed renewal rate of untreated customers from the observed renewal rate of treated customers.

This comparison is not necessarily causal because treatment assignment depends on customer characteristics.

In [3]:
treated_mask = observed_df["treatment"] == 1
control_mask = observed_df["treatment"] == 0


treated_count = treated_mask.sum()
control_count = control_mask.sum()

treated_outcome_rate = observed_df.loc[
    treated_mask,
    "outcome",
].mean()

control_outcome_rate = observed_df.loc[
    control_mask,
    "outcome",
].mean()


naive_ate = (
    treated_outcome_rate
    - control_outcome_rate
)

naive_bias = (
    naive_ate
    - true_probability_ate
)


assert treated_count + control_count == len(observed_df)
assert 0 <= treated_outcome_rate <= 1
assert 0 <= control_outcome_rate <= 1


print("Treated customers:", treated_count)
print("Control customers:", control_count)

print(
    "Treated outcome rate:",
    treated_outcome_rate,
)

print(
    "Control outcome rate:",
    control_outcome_rate,
)

print("Naive ATE estimate:", naive_ate)

print(
    "True probability-scale ATE:",
    true_probability_ate,
)

print(
    "Naive estimation bias:",
    naive_bias,
)

Treated customers: 1698
Control customers: 4302
Treated outcome rate: 0.5559481743227326
Control outcome rate: 0.4153881915388192
Naive ATE estimate: 0.1405599827839134
True probability-scale ATE: 0.15836461343754135
Naive estimation bias: -0.017804630653627945


## 4. Inverse Probability Weighting

The propensity score is the probability of receiving treatment conditional on observed pre-treatment characteristics.

A logistic regression model is used to estimate this probability. Treatment and outcome are not included among the predictors.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


causal_numeric_features = [
    "age",
    "tenure",
    "spend",
    "complaints",
    "satisfaction",
    "engagement",
]

causal_categorical_features = [
    "region",
]

causal_features = (
    causal_numeric_features
    + causal_categorical_features
)


assert "treatment" not in causal_features
assert "outcome" not in causal_features


propensity_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            causal_numeric_features,
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
            causal_categorical_features,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)


propensity_model = Pipeline(
    steps=[
        (
            "preprocessing",
            propensity_preprocessor,
        ),
        (
            "logistic_regression",
            LogisticRegression(
                max_iter=1500,
                random_state=42,
            ),
        ),
    ]
)


print("Causal numeric features:", causal_numeric_features)
print(
    "Causal categorical features:",
    causal_categorical_features,
)
print("All causal features:", causal_features)
print("Number of causal features:", len(causal_features))
print(propensity_model)

Causal numeric features: ['age', 'tenure', 'spend', 'complaints', 'satisfaction', 'engagement']
Causal categorical features: ['region']
All causal features: ['age', 'tenure', 'spend', 'complaints', 'satisfaction', 'engagement', 'region']
Number of causal features: 7
Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                                  ['age', 'tenure', 'spend',
                                                   'complaints', 'satisfaction',
                                                   'engagement']),
                                                 ('categorical',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['region'])],
                                   verbose_feature_names_out=False)),
                ('log

In [5]:
propensity_model.fit(
    observed_df[causal_features],
    observed_df["treatment"],
)


raw_propensity = propensity_model.predict_proba(
    observed_df[causal_features]
)[:, 1]

estimated_propensity = np.clip(
    raw_propensity,
    0.03,
    0.97,
)


treatment_array = observed_df[
    "treatment"
].to_numpy(dtype=float)

outcome_array = observed_df[
    "outcome"
].to_numpy(dtype=float)


ipw_contribution = (
    treatment_array
    * outcome_array
    / estimated_propensity
    -
    (1 - treatment_array)
    * outcome_array
    / (1 - estimated_propensity)
)

ipw_ate = ipw_contribution.mean()

ipw_bias = (
    ipw_ate
    - true_probability_ate
)


ipw_weights = (
    treatment_array
    / estimated_propensity
    +
    (1 - treatment_array)
    / (1 - estimated_propensity)
)

number_clipped = np.sum(
    raw_propensity != estimated_propensity
)


assert len(estimated_propensity) == len(observed_df)
assert np.isfinite(ipw_contribution).all()
assert np.isfinite(ipw_weights).all()

assert (
    (estimated_propensity >= 0.03)
    & (estimated_propensity <= 0.97)
).all()


print(
    "Minimum raw propensity:",
    raw_propensity.min(),
)

print(
    "Maximum raw propensity:",
    raw_propensity.max(),
)

print(
    "Mean estimated propensity:",
    estimated_propensity.mean(),
)

print(
    "Observed treatment rate:",
    treatment_array.mean(),
)

print(
    "Number of clipped propensities:",
    number_clipped,
)

print(
    "Maximum IPW weight:",
    ipw_weights.max(),
)

print("IPW ATE estimate:", ipw_ate)

print(
    "True probability-scale ATE:",
    true_probability_ate,
)

print("IPW estimation bias:", ipw_bias)

Minimum raw propensity: 0.12795049072269607
Maximum raw propensity: 0.6820247765176907
Mean estimated propensity: 0.2830432082276782
Observed treatment rate: 0.283
Number of clipped propensities: 0
Maximum IPW weight: 7.253359146455022
IPW ATE estimate: 0.15608880434572875
True probability-scale ATE: 0.15836461343754135
IPW estimation bias: -0.002275809091812603


## 5. Cross-Fitted T-Learner

The T-learner fits separate outcome models for treated and untreated customers.

Cross-fitting ensures that each customer's potential outcomes are predicted by models that were trained without using that customer.

In [6]:
from sklearn.base import clone
from sklearn.model_selection import KFold


base_outcome_model = Pipeline(
    steps=[
        (
            "preprocessing",
            clone(propensity_preprocessor),
        ),
        (
            "logistic_regression",
            LogisticRegression(
                max_iter=1500,
                random_state=42,
            ),
        ),
    ]
)


folds = KFold(
    n_splits=3,
    shuffle=True,
    random_state=42,
)


mu0_hat_t = np.zeros(len(observed_df))
mu1_hat_t = np.zeros(len(observed_df))


for fold_number, (train_index, test_index) in enumerate(
    folds.split(observed_df),
    start=1,
):
    fold_train = observed_df.iloc[train_index]
    fold_test = observed_df.iloc[test_index]

    fold_control = (
        fold_train["treatment"] == 0
    )

    fold_treated = (
        fold_train["treatment"] == 1
    )

    outcome_model_0 = clone(
        base_outcome_model
    )

    outcome_model_1 = clone(
        base_outcome_model
    )

    outcome_model_0.fit(
        fold_train.loc[
            fold_control,
            causal_features,
        ],
        fold_train.loc[
            fold_control,
            "outcome",
        ],
    )

    outcome_model_1.fit(
        fold_train.loc[
            fold_treated,
            causal_features,
        ],
        fold_train.loc[
            fold_treated,
            "outcome",
        ],
    )

    mu0_hat_t[test_index] = (
        outcome_model_0.predict_proba(
            fold_test[causal_features]
        )[:, 1]
    )

    mu1_hat_t[test_index] = (
        outcome_model_1.predict_proba(
            fold_test[causal_features]
        )[:, 1]
    )

    print(
        f"Fold {fold_number}:",
        f"train={len(fold_train)},",
        f"prediction={len(fold_test)}",
    )


t_learner_cate_hat = (
    mu1_hat_t
    - mu0_hat_t
)

t_learner_ate = (
    t_learner_cate_hat.mean()
)

t_learner_bias = (
    t_learner_ate
    - true_probability_ate
)


assert np.isfinite(mu0_hat_t).all()
assert np.isfinite(mu1_hat_t).all()

assert (
    (mu0_hat_t >= 0)
    & (mu0_hat_t <= 1)
).all()

assert (
    (mu1_hat_t >= 0)
    & (mu1_hat_t <= 1)
).all()


print(
    "Mean predicted outcome without treatment:",
    mu0_hat_t.mean(),
)

print(
    "Mean predicted outcome with treatment:",
    mu1_hat_t.mean(),
)

print(
    "Minimum estimated CATE:",
    t_learner_cate_hat.min(),
)

print(
    "Maximum estimated CATE:",
    t_learner_cate_hat.max(),
)

print(
    "T-learner ATE estimate:",
    t_learner_ate,
)

print(
    "True probability-scale ATE:",
    true_probability_ate,
)

print(
    "T-learner estimation bias:",
    t_learner_bias,
)

Fold 1: train=4000, prediction=2000
Fold 2: train=4000, prediction=2000
Fold 3: train=4000, prediction=2000
Mean predicted outcome without treatment: 0.40466189923815543
Mean predicted outcome with treatment: 0.5647183929511229
Minimum estimated CATE: -0.1582946975075663
Maximum estimated CATE: 0.4505219658340992
T-learner ATE estimate: 0.16005649371296754
True probability-scale ATE: 0.15836461343754135
T-learner estimation bias: 0.0016918802754261886


## 6. Cross-Fitted AIPW

Augmented inverse probability weighting combines outcome regression with propensity-score weighting.

Cross-fitting is used so that each nuisance prediction is produced by models trained without that observation.

In [7]:
aipw_folds = KFold(
    n_splits=3,
    shuffle=True,
    random_state=42,
)


raw_e_hat_aipw = np.zeros(
    len(observed_df)
)

mu0_hat_aipw = np.zeros(
    len(observed_df)
)

mu1_hat_aipw = np.zeros(
    len(observed_df)
)


for fold_number, (train_index, test_index) in enumerate(
    aipw_folds.split(observed_df),
    start=1,
):
    fold_train = observed_df.iloc[
        train_index
    ]

    fold_test = observed_df.iloc[
        test_index
    ]

    fold_control = (
        fold_train["treatment"] == 0
    )

    fold_treated = (
        fold_train["treatment"] == 1
    )


    fold_propensity_model = clone(
        propensity_model
    )

    fold_outcome_model_0 = clone(
        base_outcome_model
    )

    fold_outcome_model_1 = clone(
        base_outcome_model
    )


    fold_propensity_model.fit(
        fold_train[causal_features],
        fold_train["treatment"],
    )

    raw_e_hat_aipw[test_index] = (
        fold_propensity_model.predict_proba(
            fold_test[causal_features]
        )[:, 1]
    )


    fold_outcome_model_0.fit(
        fold_train.loc[
            fold_control,
            causal_features,
        ],
        fold_train.loc[
            fold_control,
            "outcome",
        ],
    )

    fold_outcome_model_1.fit(
        fold_train.loc[
            fold_treated,
            causal_features,
        ],
        fold_train.loc[
            fold_treated,
            "outcome",
        ],
    )


    mu0_hat_aipw[test_index] = (
        fold_outcome_model_0.predict_proba(
            fold_test[causal_features]
        )[:, 1]
    )

    mu1_hat_aipw[test_index] = (
        fold_outcome_model_1.predict_proba(
            fold_test[causal_features]
        )[:, 1]
    )


    print(
        f"Fold {fold_number}:",
        f"train={len(fold_train)},",
        f"prediction={len(fold_test)}",
    )


e_hat_aipw = np.clip(
    raw_e_hat_aipw,
    0.03,
    0.97,
)


aipw_score = (
    mu1_hat_aipw
    - mu0_hat_aipw
    + treatment_array
    * (
        outcome_array
        - mu1_hat_aipw
    )
    / e_hat_aipw
    - (1 - treatment_array)
    * (
        outcome_array
        - mu0_hat_aipw
    )
    / (1 - e_hat_aipw)
)


aipw_ate = aipw_score.mean()

aipw_standard_error = (
    aipw_score.std(ddof=1)
    / np.sqrt(len(aipw_score))
)

aipw_ci_lower = (
    aipw_ate
    - 1.96 * aipw_standard_error
)

aipw_ci_upper = (
    aipw_ate
    + 1.96 * aipw_standard_error
)

aipw_bias = (
    aipw_ate
    - true_probability_ate
)

aipw_number_clipped = np.sum(
    raw_e_hat_aipw != e_hat_aipw
)


assert np.isfinite(aipw_score).all()
assert np.isfinite(mu0_hat_aipw).all()
assert np.isfinite(mu1_hat_aipw).all()
assert np.isfinite(e_hat_aipw).all()

assert (
    (e_hat_aipw >= 0.03)
    & (e_hat_aipw <= 0.97)
).all()


print(
    "Minimum cross-fitted propensity:",
    e_hat_aipw.min(),
)

print(
    "Maximum cross-fitted propensity:",
    e_hat_aipw.max(),
)

print(
    "Number of clipped propensities:",
    aipw_number_clipped,
)

print("AIPW ATE estimate:", aipw_ate)

print(
    "AIPW standard error:",
    aipw_standard_error,
)

print(
    "AIPW 95% confidence interval:",
    (aipw_ci_lower, aipw_ci_upper),
)

print(
    "True probability-scale ATE:",
    true_probability_ate,
)

print(
    "AIPW estimation bias:",
    aipw_bias,
)

Fold 1: train=4000, prediction=2000
Fold 2: train=4000, prediction=2000
Fold 3: train=4000, prediction=2000
Minimum cross-fitted propensity: 0.12068249858766407
Maximum cross-fitted propensity: 0.6859910219551432
Number of clipped propensities: 0
AIPW ATE estimate: 0.15903890917743183
AIPW standard error: 0.014461268377706078
AIPW 95% confidence interval: (np.float64(0.1306948231571279), np.float64(0.18738299519773574))
True probability-scale ATE: 0.15836461343754135
AIPW estimation bias: 0.0006742957398904736


## 7. Estimator Comparison

The four ATE estimators are compared against the simulated probability-scale ground truth.

Signed bias indicates the direction of error, while absolute bias measures the distance from the true effect.

In [8]:
estimator_comparison = pd.DataFrame(
    {
        "estimator": [
            "Naive",
            "IPW",
            "T-Learner",
            "AIPW",
        ],
        "ate_estimate": [
            naive_ate,
            ipw_ate,
            t_learner_ate,
            aipw_ate,
        ],
        "standard_error": [
            np.nan,
            np.nan,
            np.nan,
            aipw_standard_error,
        ],
        "ci_lower_95": [
            np.nan,
            np.nan,
            np.nan,
            aipw_ci_lower,
        ],
        "ci_upper_95": [
            np.nan,
            np.nan,
            np.nan,
            aipw_ci_upper,
        ],
    }
)


estimator_comparison[
    "true_probability_ate"
] = true_probability_ate

estimator_comparison[
    "signed_bias"
] = (
    estimator_comparison["ate_estimate"]
    - estimator_comparison["true_probability_ate"]
)

estimator_comparison[
    "absolute_bias"
] = estimator_comparison[
    "signed_bias"
].abs()

estimator_comparison[
    "ate_percentage_points"
] = (
    estimator_comparison["ate_estimate"]
    * 100
)


estimator_comparison = (
    estimator_comparison.sort_values(
        "absolute_bias"
    )
    .reset_index(drop=True)
)


REPORTS_TABLES_DIR = (
    PROJECT_ROOT
    / "reports"
    / "tables"
)

REPORTS_TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

comparison_path = (
    REPORTS_TABLES_DIR
    / "ate_estimator_comparison.csv"
)

estimator_comparison.to_csv(
    comparison_path,
    index=False,
)


assert len(estimator_comparison) == 4
assert estimator_comparison[
    "ate_estimate"
].notna().all()

assert estimator_comparison[
    "absolute_bias"
].notna().all()

assert comparison_path.exists()


display(estimator_comparison)

print(
    "Best estimator by absolute bias:",
    estimator_comparison.loc[
        0,
        "estimator",
    ],
)

print(
    "Smallest absolute bias:",
    estimator_comparison.loc[
        0,
        "absolute_bias",
    ],
)

print(
    "Results saved to:",
    comparison_path,
)

print(
    "Results file exists:",
    comparison_path.exists(),
)

print(
    "All ATE estimator checks passed."
)

,estimator,ate_estimate,standard_error,ci_lower_95,ci_upper_95,true_probability_ate,signed_bias,absolute_bias,ate_percentage_points
0,AIPW,0.159039,0.014461,0.130695,0.187383,0.158365,0.000674,0.000674,15.903891
1,T-Learner,0.160056,NaN,NaN,NaN,0.158365,0.001692,0.001692,16.005649
2,IPW,0.156089,NaN,NaN,NaN,0.158365,-0.002276,0.002276,15.608880
3,Naive,0.140560,NaN,NaN,NaN,0.158365,-0.017805,0.017805,14.055998


Best estimator by absolute bias: AIPW
Smallest absolute bias: 0.0006742957398904736
Results saved to: D:\article28-unet-projects\predictive-vs-causal-decision-lab\reports\tables\ate_estimator_comparison.csv
Results file exists: True
All ATE estimator checks passed.
